# Branching Visualization

Visualize the first 2 levels of the B&B tree (root + children) using OrderRoot.

In [ ]:
import sys

sys.path.insert(0, "../../examples")

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from _utils import generate_random_instance
from shapely.plotting import plot_line, plot_polygon
from tspn_bnb2 import solve_annotated_instance
from tspn_bnb2.operations import to_shapely_linestring
from tspn_bnb2.order_annotation import add_order_annotations

In [ ]:
instance = generate_random_instance(num_polygons=15, seed=1234)
instance = add_order_annotations(instance)

In [ ]:
# Run solver and capture root node reference
captured_root = {}


def capture_root(context):
    if context.current_node.depth() == 0 and "root" not in captured_root:
        captured_root["root"] = context.root_node


solve_annotated_instance(instance, timelimit=10, root="OrderRoot", callback=capture_root)

# Extract root data
root_node = captured_root["root"]
root_seq = root_node.get_fixed_sequence()
root_indices = {te.geo_index() for te in root_seq}
root_traj = to_shapely_linestring(root_node.get_relaxed_solution().get_trajectory())

# Get ALL children (including pruned) via the new binding
children_data = []
for child in root_node.get_children():
    seq = child.get_fixed_sequence()
    indices = {te.geo_index() for te in seq}
    added = indices - root_indices
    added_idx = added.pop() if added else None
    traj = to_shapely_linestring(child.get_relaxed_solution().get_trajectory())
    children_data.append(
        {
            "trajectory": traj,
            "indices": indices,
            "added_index": added_idx,
            "pruned": child.is_pruned(),
        }
    )

print(f"Root indices: {root_indices}")
print(f"Children (total): {len(children_data)}")
for i, c in enumerate(children_data):
    status = " (pruned)" if c["pruned"] else ""
    print(f"  Child {i}: + poly {c['added_index']}{status}")

In [ ]:
n_children = len(children_data)
n_cols = max(n_children, 1)

fig = plt.figure(figsize=(2 * n_cols, 4))

# Two separate gridspecs: 1 column for root (centered), n_cols for children
gs_root = fig.add_gridspec(1, 1, left=0.35, right=0.65, top=0.95, bottom=0.55)
gs_children = fig.add_gridspec(1, n_cols, left=0.02, right=0.98, top=0.45, bottom=0.02)

GRAY = (0.8, 0.8, 0.8, 0.3)
HIGHLIGHT = (0.25, 0.55, 0.85, 0.6)

# Collect all polygon indices that get added in any child
all_added = {c["added_index"] for c in children_data if c["added_index"] is not None}


def plot_instance(ax, traj, poly_list, highlight_indices=None):
    """Plot polygons and trajectory. Highlighted indices get color, rest gray."""
    if highlight_indices is None:
        highlight_indices = set()
    for j, poly in enumerate(poly_list):
        fc = HIGHLIGHT if j in highlight_indices else GRAY
        plot_polygon(poly, ax=ax, facecolor=fc, edgecolor="black", add_points=False)
    if traj is not None:
        plot_line(traj, ax=ax, color="black", linewidth=1.5)
    ax.set_aspect("equal")
    ax.set_axis_off()


# --- Root node (centered) --- highlight the branching polygon(s)
ax_root = fig.add_subplot(gs_root[0, 0])
plot_instance(ax_root, root_traj, instance.polygons, all_added)

# --- Children ---
child_axes = []
for i, child in enumerate(children_data):
    ax = fig.add_subplot(gs_children[0, i])
    child_axes.append(ax)
    highlight = {child["added_index"]} if child["added_index"] is not None else set()
    plot_instance(ax, child["trajectory"], instance.polygons, highlight)

# --- Draw arrows from root to each child ---
fig.canvas.draw()
root_bbox = ax_root.get_position()
root_bottom_x = root_bbox.x0 + root_bbox.width / 2
root_bottom_y = root_bbox.y0

for ax in child_axes:
    child_bbox = ax.get_position()
    child_top_x = child_bbox.x0 + child_bbox.width / 2
    child_top_y = child_bbox.y1

    fig.add_artist(
        mpatches.FancyArrowPatch(
            (root_bottom_x, root_bottom_y),
            (child_top_x, child_top_y),
            transform=fig.transFigure,
            arrowstyle="-|>",
            mutation_scale=15,
            color="black",
            linewidth=1.2,
            connectionstyle="arc3,rad=0",
        )
    )

plt.savefig("out/branching.pdf", bbox_inches="tight", facecolor="white")
plt.show()